# Test grafico di `metricEuclidean`
- $V_0 < 0$: collision
- $V_0 > 0$: no collision
- $V_0 = 0$: contact

In [1]:
from pathlib import Path
import json
import sys

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import TwoSlopeNorm

project_root = Path.cwd()
if not (project_root / "hj_reachability").is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import hj_reachability as hj
from hj_reachability.vehicle.geometry import build_terminal_set_boundary

## 1. Load BRT

In [2]:
BRT_FILENAME = "brt_euclidean_2.npz"
results_directory = project_root / "results"
result_path = results_directory / BRT_FILENAME

if not result_path.is_file():
    raise FileNotFoundError(result_path)

data = np.load(result_path, allow_pickle=False)
metadata = json.loads(str(data["metadata_json"].item()))

V0 = data["V0"]
BRT = data["BRT"]
gradients_key = "gradients" if "gradients" in data.files else "gradienti"
gradients = data[gradients_key]

grid_lo = data["grid_lo"]
grid_hi = data["grid_hi"]
grid_shape = tuple(int(value) for value in data["grid_shape"])
periodic_dims = tuple(int(value) for value in data["periodic_dims"])

grid = hj.Grid.from_lattice_parameters_and_boundary_conditions(
    domain=hj.sets.Box(lo=jnp.asarray(grid_lo), hi=jnp.asarray(grid_hi)),
    shape=grid_shape,
    periodic_dims=periodic_dims,
)

print("File caricato:", result_path.resolve())
print("Forma griglia:", grid.shape)
print("Dimensioni periodiche:", periodic_dims)
print("Tempo finale:", float(data["target_time"]))

File caricato: /home/alessio/TESI/hj_reachability/results/brt_euclidean_2.npz
Forma griglia: (26, 13, 15, 8, 21, 8)
Dimensioni periodiche: ()
Tempo finale: -3.0


## 2. Dimensions check

In [3]:
expected_shape = tuple(grid.shape)
expected_gradient_shape = (*expected_shape, 6)

assert periodic_dims == (), f"La griglia non è non-periodica: {periodic_dims}"
assert V0.shape == expected_shape
assert BRT.shape == expected_shape
assert gradients.shape == expected_gradient_shape
assert tuple(metadata["grid"]["shape"]) == expected_shape
assert tuple(metadata["grid"]["periodic_dims"]) == periodic_dims
assert np.isfinite(V0).all()
assert np.isfinite(BRT).all()
assert np.isfinite(gradients).all()

print("Controlli superati.")
print(f"V0:        [{V0.min():.6f}, {V0.max():.6f}]")
print(f"BRT:       [{BRT.min():.6f}, {BRT.max():.6f}]")
print(f"Gradienti: [{gradients.min():.6f}, {gradients.max():.6f}]")
print(json.dumps(metadata, indent=2))

Controlli superati.
V0:        [-3.204360, 13.482062]
BRT:       [-50.869579, 13.482062]
Gradienti: [-318.877808, 318.810333]
{
  "created_at": "2026-08-18T19:37:43.793860+02:00",
  "state_names": [
    "x_rel",
    "y_rel",
    "theta_rel",
    "v_H",
    "delta_E",
    "v_E"
  ],
  "grid": {
    "lo": [
      -8.0,
      -6.0,
      -0.7853981852531433,
      1.0,
      -0.2617993950843811,
      1.0
    ],
    "hi": [
      17.0,
      6.0,
      0.7853981852531433,
      11.0,
      0.2617993950843811,
      11.0
    ],
    "shape": [
      26,
      13,
      15,
      8,
      21,
      8
    ],
    "spacing": [
      1.0,
      1.0,
      0.11219974075044904,
      1.4285714285714286,
      0.02617993950843811,
      1.4285714285714286
    ],
    "periodic_dims": [],
    "total_points": 6814080
  },
  "terminal_set": {
    "metric": "metricEuclidean",
    "n_theta": 90,
    "n_phi": 180
  },
  "dynamics": {
    "class": "RelativeVehicle6D",
    "module": "hj_reachability.systems

## 3. Slice selection

In [4]:
x_axis = data["x_rel"]
y_axis = data["y_rel"]
theta_axis = data["theta_rel"]
v_H_axis = data["v_H"]
delta_E_axis = data["delta_E"]
v_E_axis = data["v_E"]

theta_target = np.deg2rad(0.0)
v_H_target = 6.0
delta_E_target = np.deg2rad(0.0)
v_E_target = 6.0

i_theta = int(np.argmin(np.abs(theta_axis - theta_target)))
i_v_H = int(np.argmin(np.abs(v_H_axis - v_H_target)))
i_delta_E = int(np.argmin(np.abs(delta_E_axis - delta_E_target)))
i_v_E = int(np.argmin(np.abs(v_E_axis - v_E_target)))

selected = {
    "theta_rel [deg]": np.rad2deg(theta_axis[i_theta]),
    "v_H [m/s]": v_H_axis[i_v_H],
    "delta_E [deg]": np.rad2deg(delta_E_axis[i_delta_E]),
    "v_E [m/s]": v_E_axis[i_v_E],
}

for name, value in selected.items():
    print(f"{name}: {float(value):.6f}")

V0_slice = V0[:, :, i_theta, i_v_H, i_delta_E, i_v_E]
BRT_slice = BRT[:, :, i_theta, i_v_H, i_delta_E, i_v_E]
x_plot, y_plot = np.meshgrid(x_axis, y_axis, indexing="ij")

theta_rel [deg]: 0.000001
v_H [m/s]: 5.285715
delta_E [deg]: 0.000000
v_E [m/s]: 5.285715


## 4. Plot

In [6]:
import plotly.graph_objects as go

BRT_3D = BRT[:, :, :, i_v_H, i_delta_E, i_v_E]
X, Y, THETA = np.meshgrid(x_axis, y_axis, theta_axis, indexing="ij")

figure = go.Figure(
    data=go.Isosurface(
        x=X.ravel(),
        y=Y.ravel(),
        z=np.rad2deg(THETA.ravel()),
        value=BRT_3D.ravel(),
        isomin=0.0,
        isomax=0.0,
        surface_count=1,
        caps=dict(x_show=False, y_show=False, z_show=False),
        colorscale="Reds",
        showscale=False,
    )
)

figure.update_layout(
    title=(
        f"BRT: v_H={float(v_H_axis[i_v_H]):.3f} m/s, "
        f"delta_E={np.rad2deg(delta_E_axis[i_delta_E]):.1f} deg, "
        f"v_E={float(v_E_axis[i_v_E]):.3f} m/s"
    ),
    scene=dict(
        xaxis_title="x_rel [m]",
        yaxis_title="y_rel [m]",
        zaxis_title="theta_rel [deg]",
        aspectmode="manual",
        aspectratio=dict(x=2.0, y=1.0, z=0.9),
    ),
    width=950,
    height=700,
)

figure.show(renderer="browser")